<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week9_fairness_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before running any fairness metric: identify which customer segment you are most concerned about. Why might the model perform differently for that group? Write your hypothesis before seeing the numbers.

In [ ]:
# ── Colab setup (skip if running locally) ────────────────────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/data-science-internshipinabook.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 9 — Ethics and Fairness
### Week 9 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

The Data Science Internship Book 0 of 9 in the InternshipInABook™ Series

This repository contains the practical exercises from the book.

📖 Complete internship experience:

Workplace scenarios
Mentor guidance
Reflection exercises
Portfolio
checkpoints
Capstone projects
👉 Get the complete book: https://selar.com/al990ay7ux

🚀 Start Here
Welcome to The Data Science Internship.

If you are new to the InternshipInABook™ Series:

Run this setup notebook.
Complete Week 1.
Follow the weekly internship schedule.
Build your portfolio.
Complete the capstone project.
🔗 Repository: https://github.com/internshipinabook/data-science-internshipinabook

Run every cell top to bottom. Each cell prints ✅ if everything is working or ❌ with a fix instruction. Complete every fix before moving to Week 1.

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Reporting only overall accuracy in the model card | A model that is accurate overall can be unjust for a specific group. Fairness requires disaggregated metrics. | Always report per-group precision and recall alongside overall metrics |
| Treating the fairness audit as a box to tick | The audit has no value if you do not act on it. Every disparity requires a mitigation plan — even if the plan is 'monitor closely'. | Every finding in the fairness section must have a corresponding mitigation entry |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Compute demographic parity difference, equalised odds difference, and per-group F1 for at least two customer segments. Which metric matters most for this use case and why?

> 🔬 **Advanced Path**
> Propose one concrete mitigation for the largest disparity you found. Implement it (resampling, threshold adjustment, or reweighting) and show the before/after fairness metrics.

## ✅ What You Can Do After This Week

- Define demographic parity, equal opportunity, and equalised odds — and choose between them
- Compute equal opportunity difference and FPR parity difference per subgroup
- Identify fairness gaps exceeding 10 percentage points and propose mitigation
- Calculate the cost-of-fairness tradeoff in business terms
- Write an ethics brief documenting audit findings and monitoring commitments

*Add `week9_fairness.ipynb` to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week9_fairness_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week9/week9_fairness_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook is written for the Telco Churn dataset but the **entire ML pipeline works on any binary classification problem**.

**To swap in your own data, change only these lines at the top of the notebook:**
```python
DATA_FILE      = 'customer_churn.csv'  # ← your CSV filename
TARGET_COL     = 'Churn'              # ← the binary column you want to predict (Yes/No or 1/0)
POSITIVE_LABEL = 'Yes'                # ← which value means "positive" (the event you care about)
ID_COL         = 'customerID'         # ← identifier column to drop before modelling (or None)
NUMERIC_FIX    = 'TotalCharges'       # ← any numeric column stored as text (or None)
```

**Works with any binary classification dataset:**
- 🏦 Loan default prediction
- 🏥 Patient readmission risk
- 📧 Email spam detection
- 💳 Credit card fraud
- 🛒 Customer conversion
- 🎓 Student dropout prediction

> ⚠️ **Class imbalance warning:** If your dataset has <20% positive class, use `class_weight='balanced'` — already done in this notebook. Check your class distribution in Step 4 before proceeding.

---

## 🔄 Using a Different Dataset?

This notebook is written for `customer_churn.csv` but the ML pipeline applies to any binary classification problem.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Set `TARGET_COL` to your binary target column (the thing you want to predict)
3. Update `POSITIVE_LABEL` to the positive class name (e.g. `'Yes'`, `'1'`, `'Fraud'`)
4. The `prepare_dataset()` function encodes features — update the column references to match yours

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE      = 'customer_churn.csv'   # ← change to your file
TARGET_COL     = 'Churn'               # ← your binary target column
POSITIVE_LABEL = 'Yes'                 # ← the positive class value
ID_COL         = 'customerID'          # ← identifier column to drop (or None)
```

**Works with any binary classification dataset** — fraud detection, disease prediction,
loan default, email spam, customer conversion — as long as your target has two classes.

---

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix
%matplotlib inline; sns.set_theme(style='whitegrid')
OPTIMAL_THRESHOLD=0.35; OFFER_COST=30; CLV_SAVED=240
print(f'✅ Libraries loaded')

In [ ]:
def prepare_dataset(df):
    df=df.copy()
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
    df['TotalCharges']=df['TotalCharges'].fillna(0)  # new customers have zero charges
    if 'customerID' in df.columns: df=df.drop(columns=['customerID'])
    le=LabelEncoder(); df['Churn']=le.fit_transform(df['Churn'].astype(str))
    df['tenure_charge_ratio']=df['TotalCharges']/(df['tenure']+1)
    sc=[c for c in ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                     'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
        if c in df.columns]
    for col in sc: df[f'{col}_binary']=df[col].apply(lambda x:1 if str(x).lower()=='yes' else 0)
    df['num_services']=df[[f'{c}_binary' for c in sc]].sum(axis=1)
    df['contract_numeric']=df['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})
    df['paperless_numeric']=(df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction']=df['contract_numeric']*df['paperless_numeric']
    df['tenure_segment']=pd.cut(df['tenure'],bins=[0,12,36,float('inf')],labels=['New','Established','Loyal'])
    return df
print('✅ prepare_dataset() defined')

## Fairness Framework (write before any code)

**Potential harm:**

**Appropriate fairness definition:**

**Segments to examine:**

**'Fair enough' threshold:**

## 1. Load, prepare, preserve sensitive columns BEFORE encoding

In [ ]:
# YOUR CODE HERE
# sensitive_cols = ['gender','Contract','tenure_segment']
# df_sensitive = df_prep[sensitive_cols + ['Churn']].copy()


## 2. Train, predict, then — INDEX ALIGNMENT (CRITICAL)

In [ ]:
# After train_test_split and predictions:
# ── INDEX ALIGNMENT ──────────────────────────────────────────────────────────
# y_test retains original DataFrame index; y_pred is zero-indexed.
# Misalignment causes WRONG subgroup assignments without any error.
# Fix: reset to zero-based before any subgroup analysis.

# y_test_arr     = y_test.reset_index(drop=True).values
# sensitive_test = sensitive_test.reset_index(drop=True)
# Then verify: print(len(y_test_arr), len(sensitive_test), len(y_pred), len(y_proba))

# YOUR CODE HERE — full setup, train, predict


## 3. subgroup_performance() function (with min_group_size guard)

In [ ]:
def subgroup_performance(y_true, y_pred, y_proba, groups, min_group_size=30):
    """All arrays must be zero-indexed before calling."""
    # YOUR CODE HERE
    pass


## 4. Analysis 1 — Contract Type (table, recall gap, bar chart)

In [ ]:
# YOUR CODE HERE


## 5. Analysis 2 — Tenure Segment (table, recall gap, bar chart)

In [ ]:
# YOUR CODE HERE


## 6. Analysis 3 — Gender (table, recall gap, interpretation)

In [ ]:
# YOUR CODE HERE


## 7. Three Fairness Metric Functions

In [ ]:
def demographic_parity_difference(y_pred, groups):
    # YOUR CODE HERE
    pass

def equal_opportunity_difference(y_true, y_pred, groups):
    # YOUR CODE HERE
    pass

def fpr_parity_difference(y_true, y_pred, groups):
    # YOUR CODE HERE
    pass


## 8. Apply all three metrics to contract type and tenure segment

In [ ]:
# YOUR CODE HERE


## 9. Fairness landscape chart (two panels, 10pp orange threshold line)

In [ ]:
# YOUR CODE HERE


## 10. group_threshold_analysis() — post-processing mitigation

In [ ]:
def group_threshold_analysis(y_true, y_proba, groups,
                              default_threshold=0.35, target_recall=0.74):
    """Find per-group threshold achieving target recall. Returns dict keyed by group name."""
    # YOUR CODE HERE
    pass


## 11. Business cost of fairness intervention (use returned dict directly)

In [ ]:
# YOUR CODE HERE


## 12. Six-Section Ethics Brief

### 1 — Model Purpose and Scope
*(Include: 'This model is for retention offers only.')*

### 2 — Data Sources and Known Limitations

### 3 — Fairness Audit Results
*(State the finding plainly — not 'disparity' but who is affected and how.)*

### 4 — Mitigation Taken

### 5 — Remaining Risks
*(Include scope creep risk explicitly.)*

### 6 — Monitoring and Review
*(Name the specific metric and the specific trigger threshold.)*

---
## ✅ Week 9 Complete
**Next:** `week10/*_STARTER.ipynb`

---
*github.com/InternshipInABook*